# Chess: Turns vs Elo Rating Analysis

Vergleich zwischen Spieldauer (Turns) und Spieler-Elo-Ratings

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.linear_model import LinearRegression

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
# Lade Daten
df = pd.read_csv('games.csv')

# Berechne durchschnittliches Elo
df['avg_elo'] = (df['white_rating'] + df['black_rating']) / 2

print(f"Datensatz: {len(df)} Spiele")
print(f"\nBasic Info:")
print(df[['turns', 'white_rating', 'black_rating', 'avg_elo']].describe())

In [ ]:
# Korrelations-Analyse
corr_pearson = df['avg_elo'].corr(df['turns'])
corr_spearman, p_spearman = stats.spearmanr(df['avg_elo'], df['turns'])

print("\n=== KORRELATION: Turns vs Elo ===")
print(f"Pearson Korrelation: {corr_pearson:.4f}")
print(f"Spearman Korrelation: {corr_spearman:.4f}")
print(f"P-value (Spearman): {p_spearman:.2e}")

if corr_pearson > 0:
    print("✓ Positive Korrelation: Höhere Elos → längere Spiele")
else:
    print("✗ Negative Korrelation: Höhere Elos → kürzere Spiele")

In [ ]:
# Scatter Plot: Turns vs Elo
fig, ax = plt.subplots(figsize=(12, 7))

ax.scatter(df['avg_elo'], df['turns'], alpha=0.3, s=30)

# Trend-Linie hinzufügen
X = df['avg_elo'].values.reshape(-1, 1)
y = df['turns'].values
model = LinearRegression()
model.fit(X, y)
trend_line = model.predict(X)

ax.plot(df['avg_elo'].sort_values(), 
        model.predict(df['avg_elo'].sort_values().values.reshape(-1, 1)), 
        color='red', linewidth=2, label=f'Trend (r={corr_pearson:.3f})')

ax.set_xlabel('Durchschnittliches Elo-Rating', fontsize=12)
ax.set_ylabel('Spieldauer (Turns)', fontsize=12)
ax.set_title('Beziehung: Spieler-Stärke vs Spieldauer', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Regressions-Koeffizient: {model.coef_[0]:.4f}")
print(f"Intercept: {model.intercept_:.2f}")
print(f"R² Score: {model.score(X, y):.4f}")

In [ ]:
# Kategorisiere nach Elo-Klasse
bins = [0, 1200, 1400, 1600, 1800, 2500]
labels = ['< 1200', '1200-1400', '1400-1600', '1600-1800', '> 1800']
df['elo_class'] = pd.cut(df['avg_elo'], bins=bins, labels=labels)

# Vergleiche Turns pro Elo-Klasse
elo_stats = df.groupby('elo_class', observed=True)['turns'].agg(['mean', 'median', 'std', 'count'])

print("\n=== TURNS nach Elo-Klasse ===")
print(elo_stats)

# Visualisierung
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Box Plot
df.boxplot(column='turns', by='elo_class', ax=ax1)
ax1.set_xlabel('Elo-Klasse', fontsize=11)
ax1.set_ylabel('Turns', fontsize=11)
ax1.set_title('Spieldauer nach Spieler-Stärke')
plt.sca(ax1)
plt.xticks(rotation=45)

# Bar Chart (Durchschnitt)
elo_stats['mean'].plot(kind='bar', ax=ax2, color='steelblue', alpha=0.7)
ax2.set_xlabel('Elo-Klasse', fontsize=11)
ax2.set_ylabel('Durchschnittliche Turns', fontsize=11)
ax2.set_title('Durchschnittliche Spieldauer nach Elo-Klasse')
ax2.grid(alpha=0.3, axis='y')
plt.sca(ax2)
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Vergleich: White vs Black Rating Effekt auf Turns
corr_white = df['white_rating'].corr(df['turns'])
corr_black = df['black_rating'].corr(df['turns'])

print("\n=== Einzelne Rating-Effekte ===")
print(f"White Rating vs Turns: {corr_white:.4f}")
print(f"Black Rating vs Turns: {corr_black:.4f}")

# Visualisierung
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.scatter(df['white_rating'], df['turns'], alpha=0.3, s=20, color='blue')
ax1.set_xlabel('White Rating', fontsize=11)
ax1.set_ylabel('Turns', fontsize=11)
ax1.set_title(f'White Rating vs Turns (r={corr_white:.3f})', fontsize=12)
ax1.grid(alpha=0.3)

ax2.scatter(df['black_rating'], df['turns'], alpha=0.3, s=20, color='red')
ax2.set_xlabel('Black Rating', fontsize=11)
ax2.set_ylabel('Turns', fontsize=11)
ax2.set_title(f'Black Rating vs Turns (r={corr_black:.3f})', fontsize=12)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Zusammenfassung
print("\n" + "="*60)
print("ZUSAMMENFASSUNG: Turns vs Elo-Rating")
print("="*60)
print(f"\n📊 Datensatz: {len(df)} Spiele")
print(f"🎯 Durchschnittliche Turns: {df['turns'].mean():.1f}")
print(f"🎯 Durchschnittliches Elo: {df['avg_elo'].mean():.0f}")

print(f"\n📈 Korrelation (Pearson): {corr_pearson:.4f}")
print(f"   → {'STARK' if abs(corr_pearson) > 0.5 else 'MODERAT' if abs(corr_pearson) > 0.3 else 'SCHWACH'} korreliert")

print(f"\n💡 Interpretation:")
if corr_pearson > 0.1:
    print("   Höher-bewertete Spieler spielen tendenziell LÄNGERE Spiele")
elif corr_pearson < -0.1:
    print("   Höher-bewertete Spieler spielen tendenziell KÜRZERE Spiele")
else:
    print("   Kein großer Unterschied zwischen den Elo-Klassen")

print(f"\n📋 Turns nach Elo-Klasse:")
for idx, label in enumerate(labels):
    if label in elo_stats.index:
        avg = elo_stats.loc[label, 'mean']
        count = elo_stats.loc[label, 'count']
        print(f"   {label:12s}: {avg:6.1f} Turns (n={int(count):5d})")